# 08 · Feature table for the LightGBM re-ranker (Direction 3)

**Goal:** one row per query–product pair (train, val, test) with every feature the re-ranker of `09_lgbm` may use, grouped so that 09 can drop one group at a time (ablation).

| Group | Features | Source |
| --- | --- | --- |
| `ce` | `ce_score` (expected gain), `p_E, p_S, p_C, p_I` | 07_oof: train = out-of-fold, val/test = mean of the 5 fold models |
| `ce_list` | `_rank`, `_z`, `_gap` of `ce_score` and `p_E` inside the query; query max / mean / std of `ce_score` | from `ce` |
| `lexical` | BM25 `title` / `main` (+ `_rank`, `_z`, `_gap` of `bm25_main`), query-word overlap with title / main, numbers, number + unit, color, brand in query, negation broken | 02_bm25 + rules ported from `eda/eda_audit.py` |
| `biencoder` | cosine of bge-base and all-MiniLM-L6 (`title`, `main`) + `_rank`, `_z`, `_gap` of `b2_bge_main` | 04_biencoder (zero-shot, so no leakage on train) |
| `brand_list` | share of candidates with the same brand, most frequent brand of the list, number of brands | candidate list |
| `product` | title / bullet length, which fields are present | products_clean |
| `query` | query length, has a number / unit / color / negation, number of candidates | query |

**Not used** (README, banned features): catalogue row position, `product_id` patterns, `small_version` / `large_version`, the product's labels in other queries. No feature reads a label.

**Train vs val/test scores.** Train `ce` features come from one fold model; val/test ones from the mean of 5, which is smoother and a little better (4-fold mean on val: +0.003 nDCG). Section 3 compares their distributions; the `_rank` / `_z` / `_gap` features are relative to the query and should shift less. 09 checks this with an ablation.

**Kaggle settings:** CPU, no internet. About 5 min.

**Inputs** (Add Input):
- Dataset `wm-src`, version whose `src/features.py` has `match_features`. Required.
- Notebook Output **01_preprocess** (`pairs.parquet`, `products_clean.parquet`). Required.
- Notebook Output **02_bm25** (`bm25_features.parquet`). Required.
- Notebook Output **04_biencoder** (`biencoder_features.parquet`). Required.
- Notebook Output **07_oof**, the version run with `MODE = 'assemble'` (`oof/ce_deberta_main_ce4.parquet`). Required.

**Output** (`/kaggle/working/features/`): `features.parquet` (`example_id, query_id, product_id, split, fold, esci_label, gain` + features; `fold` = `fold10 // 2` on train, −1 elsewhere), `feature_groups.json`, `single_feature_ndcg.csv`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import json, time

import numpy as np
import pandas as pd

import src.features as F
from src.data import PREPROCESS_HINT, find
from src.evaluation import query_metrics

assert hasattr(F, 'match_features'), 'wm-src is an old version: rebuild the zip (features.py with match_features) and update the input'

INPUT = '/kaggle/input'
OUT = Path('/kaggle/working/features')
CE_FILE = 'ce_deberta_main_ce4.parquet'
SAMPLE = None     # e.g. 0.05: keep 5% of the queries of each split for a quick check (outputs are then not usable)
N_FOLDS = 5
PROBS = ['p_E', 'p_S', 'p_C', 'p_I']
BANNED = {'catalog_pos', 'product_prior', 'isbn_like_id', 'small_version', 'large_version', 'product_id'}

## 1. Inputs

Every source is joined on `example_id` (one row per pair) and must cover every pair; the split of the CE file must agree with `pairs.parquet`.

In [ ]:
pairs = pd.read_parquet(find('pairs.parquet', PREPROCESS_HINT, INPUT),
                        columns=['example_id', 'query_id', 'query', 'product_id', 'esci_label', 'gain', 'split', 'fold10'])
if SAMPLE:
    q = pairs[['query_id', 'split']].drop_duplicates().groupby('split', group_keys=False).sample(frac=SAMPLE, random_state=42)
    pairs = pairs[pairs['query_id'].isin(q['query_id'])]
prod = pd.read_parquet(find('products_clean.parquet', PREPROCESS_HINT, INPUT),
                       columns=['product_id', 'title', 'brand', 'color', 'bullets',
                                'has_brand', 'has_color', 'has_bullets', 'has_description'])
bm25 = pd.read_parquet(find('bm25_features.parquet', 'Add Input -> Notebook Output -> 02_bm25', INPUT))
bienc = pd.read_parquet(find('biencoder_features.parquet', 'Add Input -> Notebook Output -> 04_biencoder', INPUT))
ce = pd.read_parquet(find(CE_FILE, "Add Input -> Notebook Output -> 07_oof (version run with MODE = 'assemble')", INPUT))

df = pairs.merge(prod, on='product_id', how='left', validate='m:1')
for src_df in (bm25, bienc, ce.drop(columns=['query_id']).rename(columns={'score': 'ce_score', 'split': 'ce_split'})):
    df = df.merge(src_df, on='example_id', how='left', validate='1:1')
assert len(df) == len(pairs) and df['title'].notna().all(), 'products missing'
inputs = ['bm25_title', 'bm25_main', 'b2_bge_title', 'b2_bge_main', 'b2_minilml6_title', 'b2_minilml6_main', 'ce_score'] + PROBS
assert df[inputs].notna().all().all(), df[inputs].isna().sum()[lambda s: s > 0]
assert (df['ce_split'] == df['split']).all(), 'CE file and pairs.parquet disagree on the split'

df['fold'] = np.where(df['split'] == 'train', df['fold10'] // 2, -1)
df = df.drop(columns=['ce_split', 'fold10']).sort_values(['query_id', 'example_id']).reset_index(drop=True)
display(df.groupby('split').agg(pairs=('example_id', 'size'), queries=('query_id', 'nunique')))

## 2. Features

`match_features` loops over the pairs in Python (a few minutes). The `has_*` flags come from `products_clean`.

In [ ]:
t0 = time.time()
df = pd.concat([df, F.match_features(df['query'], df['title'], df['brand'], df['color'], df['bullets'])], axis=1)
print(f'match features {time.time() - t0:.0f} s')

for c in ['ce_score', 'p_E', 'bm25_main', 'b2_bge_main']:
    df = df.join(F.within_query(df, c))
g = df.groupby('query_id')['ce_score']
df['q_ce_max'], df['q_ce_mean'] = g.transform('max'), g.transform('mean')
df['q_ce_std'] = np.sqrt(np.maximum((df['ce_score'] ** 2).groupby(df['query_id']).transform('mean') - df['q_ce_mean'] ** 2, 0))
df['n_cand'] = df.groupby('query_id')['example_id'].transform('size').astype(float)
df = df.join(F.brand_list_features(df['query_id'], df['brand']))
for c in ['has_brand', 'has_color', 'has_bullets', 'has_description']:
    df[c] = df[c].astype(float)

GROUPS = {
    'ce': ['ce_score'] + PROBS,
    'ce_list': ['ce_score_rank', 'ce_score_z', 'ce_score_gap', 'p_E_rank', 'p_E_z', 'p_E_gap',
                'q_ce_max', 'q_ce_mean', 'q_ce_std'],
    'lexical': ['bm25_title', 'bm25_main', 'bm25_main_rank', 'bm25_main_z', 'bm25_main_gap', 'overlap_title',
                'overlap_main', 'num_match', 'unit_match', 'color_match', 'brand_in_query', 'neg_violation'],
    'biencoder': ['b2_bge_title', 'b2_bge_main', 'b2_minilml6_title', 'b2_minilml6_main',
                  'b2_bge_main_rank', 'b2_bge_main_z', 'b2_bge_main_gap'],
    'brand_list': ['brand_share', 'top_brand', 'q_n_brands'],
    'product': ['title_words', 'bullet_words', 'has_brand', 'has_color', 'has_bullets', 'has_description'],
    'query': ['q_words', 'q_has_num', 'q_has_unit', 'q_has_color', 'q_has_neg', 'n_cand'],
}
FEATURES = [c for cols in GROUPS.values() for c in cols]
assert len(FEATURES) == len(set(FEATURES)) and not BANNED & set(FEATURES)
df[FEATURES] = df[FEATURES].astype('float32')
assert np.isfinite(df[FEATURES].to_numpy()).all(), 'NaN or inf in a feature'
print(len(FEATURES), 'features in', len(GROUPS), 'groups |', f'{time.time() - t0:.0f} s')

## 3. Checks

**Train vs val/test.** Mean and spread of the CE features per split. Raw `ce_score` / `p_E` of val/test (mean of 5 models) may be a little less spread than train (one model); the within-query `_z` features should match across splits by construction.

**Single-feature nDCG** of every feature used alone to rank (train = out-of-fold, val; test stays blind), with the better of the two signs. Expected from earlier runs: `ce_score` ≈ 0.884 on train and ≈ 0.892 on val; `bm25_title` 0.816, `bm25_main` 0.823, `b2_bge_main` 0.843, `overlap_main` ≈ 0.815 on val; features constant within a query ≈ random (0.742).

In [ ]:
stats = df.groupby('split')[['ce_score', 'p_E', 'q_ce_std', 'ce_score_z', 'p_E_gap']].agg(['mean', 'std']).round(4)
display(stats.loc[['train', 'val', 'test']])


def single_feature_ndcg(part, col):
    up = query_metrics(part, part[col].to_numpy())['ndcg'].mean()
    down = query_metrics(part, -part[col].to_numpy())['ndcg'].mean()
    return (up, '+') if up >= down else (down, '-')


group_of = {c: g for g, cols in GROUPS.items() for c in cols}
rows = []
for col in FEATURES:
    r = {'feature': col, 'group': group_of[col]}
    for s in ['train', 'val']:
        r[f'ndcg_{s}'], r[f'sign_{s}'] = single_feature_ndcg(df[df['split'] == s], col)
    rows.append(r)
single = pd.DataFrame(rows).round(4)
display(single.sort_values('ndcg_val', ascending=False).reset_index(drop=True))

v = single.set_index('feature')
expected = {('ce_score', 'train'): (0.884, 0.004), ('ce_score', 'val'): (0.892, 0.003),
            ('bm25_title', 'val'): (0.8162, 0.001), ('bm25_main', 'val'): (0.8227, 0.001),
            ('b2_bge_main', 'val'): (0.8429, 0.001), ('overlap_main', 'val'): (0.815, 0.004),
            ('n_cand', 'val'): (0.742, 0.003)}
if not SAMPLE:
    for (col, s), (ref, tol) in expected.items():
        got = v.loc[col, f'ndcg_{s}']
        print(f"{'OK   ' if abs(got - ref) <= tol else 'CHECK'} {col:13s} {s:5s} {got:.4f} (expected {ref} ± {tol})")
    print('sign mismatch train vs val:', v.index[v['sign_train'] != v['sign_val']].tolist())

## 4. Save

`features.parquet` keeps the ids, split, fold and labels next to the features; `product_id` is there for error analysis only and is not a feature. 09 reads the feature names from `feature_groups.json`.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
keep = ['example_id', 'query_id', 'product_id', 'split', 'fold', 'esci_label', 'gain'] + FEATURES
df[keep].to_parquet(OUT / 'features.parquet', index=False)
(OUT / 'feature_groups.json').write_text(json.dumps(
    {'groups': GROUPS, 'n_folds': N_FOLDS, 'ce_file': CE_FILE, 'sample': SAMPLE, 'src_version': src_version()}, indent=1))
single.to_csv(OUT / 'single_feature_ndcg.csv', index=False)
print('wrote', OUT / 'features.parquet', df[keep].shape,
      f"{(OUT / 'features.parquet').stat().st_size / 2**20:.0f} MB", '| SAMPLE =', SAMPLE)